# Loading required libraries

In [1]:
library(tidyverse)

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.1     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.3     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors


## Figures and Files Setup

### Results main directory

In [2]:
data_dir = "/Users/jawadalaaedeen/Desktop/PhD/NMC/results"

## Object for analysis

In [3]:
celltype_metadata <- read_csv(
  paste0(data_dir, "/celltype_metadata_final.csv")
) %>%
  mutate(is_lung = ifelse(tumor_site == "Lung", "Lung", "Non-Lung")) %>%
  mutate(patient_num = as.numeric(str_extract(patient_exp, "(?<=NUT_)\\d+")),
         roi_num = as.numeric(str_extract(patient_exp, "(?<=ROI)\\d+"))) %>%
  arrange(patient_num, roi_num) %>%
  select(-patient_num, -roi_num)
celltype_metadata$patient_exp <- factor(celltype_metadata$patient_exp, levels = unique(celltype_metadata$patient_exp))
celltype_metadata$patient_ID <- factor(celltype_metadata$patient_ID, levels = unique(celltype_metadata$patient_ID))
patient_ID_order <- unique(celltype_metadata$patient_ID)
patient_exp_order <- unique(celltype_metadata$patient_exp)


myeloid <-  c("M1 macrophages", "M2 macrophages", "M1/M2 macrophages", "DCs", "Granulocytes", "MDSCs", "Mast cells")
lymphoid <- c("CD4+ T cells", "CD8+ T cells", "Treg T cells", "NK cells", "B cells", "PCs")
nonimmune <- c("Actin+ cells", "Endothelial cells", "Lymphatic endothelial cells", "Epithelial cells")

celltype_metadata <- celltype_metadata %>%
  mutate(
    cell_category_parent = case_when(
      cell_category %in% c("Granulocytes", "MDSCs") ~ "Granulocytic",
      cell_category %in% c("M1 macrophages", "M2 macrophages", "M1/M2 macrophages") ~ "Macrophages",
      cell_category %in% c("CD4+ T cells", "CD8+ T cells", "Treg T cells") ~ "T cells",
      #else is as is
      TRUE ~ cell_category),
    cell_category_simplified = case_when(
      cell_category %in% myeloid ~ "Myeloid",
      cell_category %in% lymphoid ~ "Lymphoid",
      cell_category %in% nonimmune ~ "Non-immune",
      cell_category == "NFC" ~ "NFC",
      cell_category == "Tumor cells" ~ "Tumor"
    ),
    cell_category_simplified = factor(
      cell_category_simplified,
      levels = c("NFC", "Non-immune", "Lymphoid", "Myeloid", "Tumor")
    )
  ) 

Rows: 553331 Columns: 17
── Column specification ────────────────────────────────────────────────────────
Delimiter: ","
chr (10): cell_type, cell_category, patient_ID, exp_name, patient_exp, tumor...
dbl  (7): cell_id, Cell Center X, Cell Center Y, run, rois, survival_time_mo...

ℹ Use `spec()` to retrieve the full column specification for this data.
ℹ Specify the column types or set `show_col_types = FALSE` to quiet this message.


## Composition per ROI

In [4]:
#showcase proportions of celltypes per patient_exp in a stacked barplot
celltype_percentage_exp <- celltype_metadata %>%
  select(patient_exp, patient_ID, cell_category_simplified, cell_category_parent, cell_category) %>%
  group_by(patient_exp, patient_ID, cell_category_simplified, cell_category_parent, cell_category) %>%
  summarise(cell_count = n()) %>%
  ungroup() %>%
  group_by(patient_exp, patient_ID) %>%
  mutate(
    proportion = ifelse(cell_count == 0, NA, cell_count / sum(cell_count)),
    percentage = ifelse(cell_count == 0, NA, (cell_count / sum(cell_count)) * 100),
    geom_mean = exp(mean(log(proportion), na.rm = TRUE)), 
    clr = log(proportion / geom_mean)
  ) %>%
  ungroup() %>%
  mutate(patient_exp = factor(patient_exp, levels = patient_exp_order),
         patient_ID = factor(patient_ID, levels = patient_ID_order)) %>%
  arrange(patient_exp, patient_ID, cell_category_simplified,cell_category_parent, cell_category)

#write to csv to data_dir
write_csv(celltype_percentage_exp, paste0(data_dir, "/celltype_percentage_exp.csv"))

`summarise()` has regrouped the output.
ℹ Summaries were computed grouped by patient_exp, patient_ID,
  cell_category_simplified, cell_category_parent, and cell_category.
ℹ Output is grouped by patient_exp, patient_ID, cell_category_simplified, and
  cell_category_parent.
ℹ Use `summarise(.groups = "drop_last")` to silence this message.
ℹ Use `summarise(.by = c(patient_exp, patient_ID, cell_category_simplified,
  cell_category_parent, cell_category))` for per-operation grouping
  (`?dplyr::dplyr_by`) instead.


## Composition per Patient

In [5]:
#get percentages of cell types per patient
celltype_percentage_patient = celltype_percentage_exp %>%
  group_by(patient_exp, patient_ID, cell_category, cell_category_simplified) %>%
  #get the sum of the cell_category_simplified percentages
    summarise(cell_count = sum(cell_count)) %>%
    ungroup() %>%
    group_by(patient_exp, patient_ID) %>%
    mutate(
        proportion = ifelse(cell_count == 0, NA, cell_count / sum(cell_count)),
        percentage = ifelse(cell_count == 0, NA, (cell_count / sum(cell_count)) * 100),
        geom_mean = exp(mean(log(proportion), na.rm = TRUE)), 
        clr = log(proportion / geom_mean)
      ) %>%
    ungroup() %>%
    group_by(patient_ID, cell_category, cell_category_simplified) %>%
    summarise(percentage = median(percentage),
              clr = median(clr)) %>%
    ungroup() %>%
    mutate(patient_ID = factor(patient_ID, levels = patient_ID_order)) %>%
  arrange(patient_ID, cell_category_simplified, cell_category)
  
write_csv(celltype_percentage_patient, paste0(data_dir, "/celltype_percentage_patient.csv"))

`summarise()` has regrouped the output.
ℹ Summaries were computed grouped by patient_exp, patient_ID, cell_category,
  and cell_category_simplified.
ℹ Output is grouped by patient_exp, patient_ID, and cell_category.
ℹ Use `summarise(.groups = "drop_last")` to silence this message.
ℹ Use `summarise(.by = c(patient_exp, patient_ID, cell_category,
  cell_category_simplified))` for per-operation grouping (`?dplyr::dplyr_by`)
  instead.
`summarise()` has regrouped the output.
ℹ Summaries were computed grouped by patient_ID, cell_category, and
  cell_category_simplified.
ℹ Output is grouped by patient_ID and cell_category.
ℹ Use `summarise(.groups = "drop_last")` to silence this message.
ℹ Use `summarise(.by = c(patient_ID, cell_category, cell_category_simplified))`
  for per-operation grouping (`?dplyr::dplyr_by`) instead.


## Composition per Cell Parent per patient

In [6]:
#get percentages of cell types per patient
cellparent_percentage_patient = celltype_percentage_exp %>%
  group_by(patient_exp, patient_ID, cell_category_parent) %>%
  #get the sum of the cell_category_parent percentages
    summarise(cell_count = sum(cell_count)) %>%
    ungroup() %>%
    group_by(patient_exp, patient_ID) %>%
    mutate(
        proportion = ifelse(cell_count == 0, NA, cell_count / sum(cell_count)),
        percentage = ifelse(cell_count == 0, NA, (cell_count / sum(cell_count)) * 100),
        geom_mean = exp(mean(log(proportion), na.rm = TRUE)), 
        clr = log(proportion / geom_mean)
      ) %>%
    ungroup() %>%
    group_by(patient_ID, cell_category_parent) %>%
    summarise(percentage = median(percentage),
              clr = median(clr)) %>%
    ungroup() %>%
    mutate(patient_ID = factor(patient_ID, levels = patient_ID_order)) %>%
  arrange(patient_ID, cell_category_parent)
  
write_csv(cellparent_percentage_patient, paste0(data_dir, "/cellparent_percentage_patient.csv"))

`summarise()` has regrouped the output.
ℹ Summaries were computed grouped by patient_exp, patient_ID, and
  cell_category_parent.
ℹ Output is grouped by patient_exp and patient_ID.
ℹ Use `summarise(.groups = "drop_last")` to silence this message.
ℹ Use `summarise(.by = c(patient_exp, patient_ID, cell_category_parent))` for
  per-operation grouping (`?dplyr::dplyr_by`) instead.
`summarise()` has regrouped the output.
ℹ Summaries were computed grouped by patient_ID and cell_category_parent.
ℹ Output is grouped by patient_ID.
ℹ Use `summarise(.groups = "drop_last")` to silence this message.
ℹ Use `summarise(.by = c(patient_ID, cell_category_parent))` for per-operation
  grouping (`?dplyr::dplyr_by`) instead.


## Composition per Cell group per patient

In [7]:
#get percentages of cell types per patient
cellgroup_percentage_patient = celltype_percentage_exp %>%
  group_by(patient_exp, patient_ID, cell_category_simplified) %>%
  #get the sum of the cell_category_simplified percentages
    summarise(cell_count = sum(cell_count)) %>%
    ungroup() %>%
    group_by(patient_exp, patient_ID) %>%
    mutate(
        proportion = ifelse(cell_count == 0, NA, cell_count / sum(cell_count)),
        percentage = ifelse(cell_count == 0, NA, (cell_count / sum(cell_count)) * 100),
        geom_mean = exp(mean(log(proportion), na.rm = TRUE)), 
        clr = log(proportion / geom_mean)
      ) %>%
    ungroup() %>%
    group_by(patient_ID, cell_category_simplified) %>%
    summarise(percentage = median(percentage),
              clr = median(clr)) %>%
    ungroup() %>%
    mutate(patient_ID = factor(patient_ID, levels = patient_ID_order),
          cell_category_simplified = factor(cell_category_simplified, levels = c("NFC", "Non-immune", "Lymphoid", "Myeloid", "Tumor"))) %>%
  arrange(patient_ID, cell_category_simplified)
  
write_csv(cellgroup_percentage_patient, paste0(data_dir, "/cellgroup_percentage_patient.csv"))

`summarise()` has regrouped the output.
ℹ Summaries were computed grouped by patient_exp, patient_ID, and
  cell_category_simplified.
ℹ Output is grouped by patient_exp and patient_ID.
ℹ Use `summarise(.groups = "drop_last")` to silence this message.
ℹ Use `summarise(.by = c(patient_exp, patient_ID, cell_category_simplified))`
  for per-operation grouping (`?dplyr::dplyr_by`) instead.
`summarise()` has regrouped the output.
ℹ Summaries were computed grouped by patient_ID and cell_category_simplified.
ℹ Output is grouped by patient_ID.
ℹ Use `summarise(.groups = "drop_last")` to silence this message.
ℹ Use `summarise(.by = c(patient_ID, cell_category_simplified))` for
  per-operation grouping (`?dplyr::dplyr_by`) instead.
